# GCN Cloud Notebook

This notebook runs the cleaned lattice stiffness GNN workflow from `colab_gnn_stiffness_prototype.py`.
It is structured for a normal Jupyter notebook environment rather than Google Colab.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd

notebook_dir = Path.cwd().resolve()
search_roots = [notebook_dir, *notebook_dir.parents, notebook_dir / "active_projects" / "voronoi_lattice_pipeline" / "gnn_prototype"]
module_dir = next((path for path in search_roots if (path / "colab_gnn_stiffness_prototype.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError("Could not locate colab_gnn_stiffness_prototype.py")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from colab_gnn_stiffness_prototype import (
    TrainingConfig,
    SimpleGNN,
    create_data_loaders,
    default_data_roots,
    evaluate_model,
    find_pipeline_root,
    load_lattice_dataset,
    normalize_feature_splits,
    plot_prediction_splits,
    plot_training_history,
    predict_on_directory,
    save_run_artifacts,
    set_seed,
    split_dataset,
    summarize_metrics,
    train_model,
)


In [ ]:
config = TrainingConfig()
set_seed(config.seed)

train_root, predict_root = default_data_roots()
print(f"Train data: {train_root}")
print(f"Prediction data: {predict_root}")
print(f"Device: {config.device}")
print(f"Batch size: {config.batch_size}")
print(f"Hidden dim: {config.hidden_dim}")
print(f"Epochs: {config.total_epochs}")


In [ ]:
dataset = load_lattice_dataset(train_root)
train_data, val_data, test_data = split_dataset(dataset, seed=config.seed)
scaler = normalize_feature_splits(train_data, val_data, test_data)
train_loader, val_loader, test_loader = create_data_loaders(
    train_data,
    val_data,
    test_data,
    batch_size=config.batch_size,
)

model = SimpleGNN(
    input_dim=train_data[0].x.shape[1],
    hidden_dim=config.hidden_dim,
)

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")
model


In [ ]:
history = train_model(model, train_loader, val_loader, config)
plot_training_history(history, config.epochs_phase1)


In [ ]:
metrics_by_split = {}
split_results = []

for split_name, loader in (("Train", train_loader), ("Validation", val_loader), ("Test", test_loader)):
    predictions, ground_truth, metrics = evaluate_model(model, loader, device=config.device)
    metrics_by_split[split_name] = metrics
    split_results.append((split_name, predictions, ground_truth))

metrics_frame = summarize_metrics(metrics_by_split)
metrics_frame


In [ ]:
plot_prediction_splits(split_results)


In [ ]:
prediction_results, prediction_metrics = predict_on_directory(
    model,
    predict_root,
    scaler,
    device=config.device,
)

prediction_summary = pd.Series(prediction_metrics, name="Prediction Set")
display(prediction_results.head())
display(prediction_summary)

output_dir = find_pipeline_root() / "gnn_prototype" / "outputs"
save_run_artifacts(
    output_dir,
    model,
    scaler,
    history,
    metrics_by_split,
    prediction_results=prediction_results,
)
print(f"Saved artifacts to {output_dir}")
